# Partie 2 : étude principale pass@k, session 6 : lecture croisée, variante « réponse forcée » (exploratoire)

Plan gelé : `docs/partie2/plan_analyse.md` (v1.0). Amendement du 13 novembre, après la session 5 : les lectures sans réponse (le lecteur voulait chercher encore) sont complétées en imposant « <answer> » après ce que le lecteur a écrit. Sorties des sessions 1 et 5 attachées au kernel.

Notebook généré par `scripts/genere_notebooks_partie2.py` : le modifier là, pas ici.


In [ ]:
BRANCH = "partie2-passk"
N_QUESTIONS, N_SAMPLES, T, TEMP = 500, 32, 4, 0.7   # valeurs par défaut ; les options d'un run priment
MAX_TOKENS, MAX_MODEL_LEN = 1024, 8192   # figés au pilote v2 ; arrêt à </search> et </answer>
FILES = [
    [('PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-it-em-ppo-v0.3', 'qwen2.5-3b-it-em-ppo-v0.3_bridge'), ('PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-it-em-ppo-v0.3', 'Qwen2.5-3B-Instruct_bridge'), ('Qwen/Qwen2.5-3B-Instruct', 'qwen2.5-3b-it-em-ppo-v0.3_bridge'), ('Qwen/Qwen2.5-3B-Instruct', 'Qwen2.5-3B-Instruct_bridge')],
    [('PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-it-em-grpo-v0.3', 'qwen2.5-3b-it-em-grpo-v0.3_bridge'), ('PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-it-em-grpo-v0.3', 'Qwen2.5-3B-Instruct_bridge'), ('Qwen/Qwen2.5-3B-Instruct', 'qwen2.5-3b-it-em-grpo-v0.3_bridge')],
]


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
from pathlib import Path
REPO = "vqlentinn/memoire-rl-search"
CLONE = Path("/tmp/memoire-rl-search")
if not CLONE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(CLONE)], check=True)
print(subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm==0.33.0 pandas pyarrow huggingface_hub 2>&1 | tail -3
!pip freeze > /kaggle/working/pip_freeze.txt
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"
import sys
sys.path.insert(0, str(CLONE))
from data.hotpot_distractor import telecharger
PARQUET = telecharger("/tmp/hotpot")
print(PARQUET)

In [ ]:
import os, threading, time, glob
WORK = Path("/kaggle/working")
(WORK / "logs").mkdir(exist_ok=True)
status, verrou = [], threading.Lock()
COURT = {"PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-it-em-ppo-v0.3": "qwen2.5-3b-it-em-ppo-v0.3",
         "PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-it-em-grpo-v0.3": "qwen2.5-3b-it-em-grpo-v0.3",
         "Qwen/Qwen2.5-3B-Instruct": "Qwen2.5-3B-Instruct"}

def trouver(motif):
    f = sorted(glob.glob(f"/kaggle/input/**/{motif}", recursive=True))
    assert f, f"aucun fichier pour {motif}"
    return f

def file_forcee(gpu, runs):
    env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD="spawn", CUDA_VISIBLE_DEVICES=str(gpu))
    for lecteur, prefixe in runs:
        chercheur = prefixe.rsplit("_", 1)[0]
        lecture = trouver(f"lecteur-{COURT[lecteur]}_chercheur-{chercheur}_bridge_m8.jsonl")[0]
        nom = f"{COURT[lecteur]}_lit_{chercheur}_force"
        cmd = ["python", "-u", "scripts/run_lecture_forcee.py", "--lecteur", lecteur, "--lecture", lecture,
               "--chercheur", *trouver(f"{prefixe}_q500_n32_T4_temp0.7_chat*.jsonl"), "--data", PARQUET,
               "--max-model-len", str(MAX_MODEL_LEN), "--outdir", str(WORK / "results" / "partie2_lecture")]
        t0 = time.time()
        with open(WORK / "logs" / f"{nom}.log", "a") as f:
            code = subprocess.run(cmd, cwd=CLONE, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        with verrou:
            status.append((nom, gpu, code, round(time.time() - t0)))
            print(f"GPU {gpu} : {nom} code {code}, {time.time() - t0:.0f} s", flush=True)

fils = [threading.Thread(target=file_forcee, args=(g, runs)) for g, runs in enumerate(FILES)]
for f in fils: f.start()
for f in fils: f.join()


In [ ]:
import json
from collections import Counter
resume = {}
for fichier in sorted((WORK / "results" / "partie2_lecture").glob("*.jsonl")):
    L = [json.loads(l) for l in fichier.read_text().split("\n") if l.strip()]
    if not L:
        continue
    st = Counter(l["status"] for l in L)
    par_q = Counter(l["id"] for l in L)
    resume[fichier.stem] = {
        "trajectoires": len(L), "questions": len(par_q),
        "em": round(sum(l["em"] for l in L) / len(L), 3), "statuts": dict(st),
        "tronquees": round(sum(l["truncated"] for l in L) / len(L), 3),
        "recherches": round(sum(l["n_search"] for l in L) / len(L), 2),
        "jetons": round(sum(l["gen_tokens"] for l in L) / len(L)),
    }
(WORK / "resume_s6.json").write_text(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
print(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
echecs = [s for s in status if s[2] != 0]
if echecs:
    raise RuntimeError(f"Runs en échec : {echecs}")
